# Phase 9 — Hyperparameter Tuning & Imbalance Handling

## Objective

The objective of this phase is to optimize the best-performing machine learning models identified during the experimentation phase.

The selected models will undergo hyperparameter tuning to improve predictive performance. In addition, class imbalance handling techniques will be evaluated to determine whether they improve the model's ability to identify churning customers.

The optimized models will be compared against their baseline versions, and the best-performing model will be selected for deployment.

## Workflow

1. Import Required Libraries
2. Load Feature-Engineered Dataset
3. Prepare Features and Target
4. Perform Stratified Train-Test Split
5. Reproduce Baseline Performance
6. Hyperparameter Tuning
7. Handle Class Imbalance
8. Compare Optimized Models
9. Select Final Model

## Part A — Import Required Libraries

The following libraries are required for hyperparameter tuning, cross-validation, and class imbalance handling.

In [15]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

## Part B — Load Dataset

We'll reuse the helper functions from Phase 8.

In [ ]:
# # ==========================================
# # Import Libraries
# # ==========================================
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ==========================================
# # Load Feature-Engineered Dataset
# # ==========================================
# df = pd.read_csv("../data/processed/v1_feature_engineered_customer_churn.csv")

# # ==========================================
# # Separate Features and Target
# # ==========================================
# # X = df.drop(columns="Churn")
# # y = df["Churn"]

# print(df.shape)

(7032, 32)


In [16]:
def load_dataset(file_path):
    """
    Load the feature-engineered dataset.
    """

    df = pd.read_csv(file_path)

    print("=" * 60)
    print("DATASET LOADED")
    print("=" * 60)
    print(f"Shape : {df.shape}")

    return df

In [17]:
def prepare_features_target(df, target="Churn"):
    """
    Separate feature matrix and target variable.
    """

    X = df.drop(columns=target)
    y = df[target]

    print("=" * 60)
    print("FEATURE MATRIX")
    print("=" * 60)
    print(f"Features : {X.shape}")
    print(f"Target   : {y.shape}")

    return X, y

In [18]:
def split_dataset(
    X,
    y,
    test_size=0.20,
    random_state=42
):
    """
    Perform a stratified train-test split.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    print("=" * 60)
    print("TRAIN-TEST SPLIT")
    print("=" * 60)
    print(f"Training Samples : {len(X_train)}")
    print(f"Testing Samples  : {len(X_test)}")

    return X_train, X_test, y_train, y_test

In [19]:
df = load_dataset(
    "../data/processed/v1_feature_engineered_customer_churn.csv"
)

X, y = prepare_features_target(df)

X_train, X_test, y_train, y_test = split_dataset(X, y)

DATASET LOADED
Shape : (7032, 32)
FEATURE MATRIX
Features : (7032, 31)
Target   : (7032,)
TRAIN-TEST SPLIT
Training Samples : 5625
Testing Samples  : 1407


In [20]:
# Step 1 — Create a Generic Evaluation Function

def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Train and evaluate a classification model.
    """

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities (if supported)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_test)
    else:
        y_prob = None

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        pos_label=1
    )

    recall = recall_score(
        y_test,
        y_pred,
        pos_label=1
    )

    f1 = f1_score(
        y_test,
        y_pred,
        pos_label=1
    )

    if y_prob is not None:
        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )
    else:
        roc_auc = np.nan

    # Metrics
    # accuracy = accuracy_score(y_test, y_pred)
    # precision = precision_score(y_test, y_pred, pos_label="Yes")
    # recall = recall_score(y_test, y_pred, pos_label="Yes")
    # f1 = f1_score(y_test, y_pred, pos_label="Yes")

    # if y_prob is not None:
    #     roc_auc = roc_auc_score(
    #         y_test.map({"No": 0, "Yes": 1}),
    #         y_prob
    #     )
    # else:
    #     roc_auc = np.nan

    return {
        "Model": model.__class__.__name__,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "Model Object": model
    }

In [21]:
selected_models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

baseline_phase9 = []

for name, model in selected_models.items():

    print(f"Training {name}...")

    result = evaluate_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    baseline_phase9.append(result)

Training Logistic Regression...
Training Gradient Boosting...
Training AdaBoost...


## Part D — Hyperparameter Tuning

Hyperparameter tuning is performed using Grid Search with Stratified K-Fold Cross-Validation.

The search evaluates multiple combinations of hyperparameters and selects the configuration that maximizes the chosen evaluation metric.

A stratified 5-fold cross-validation strategy is used to preserve the class distribution in each fold.

In [22]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv_strategy)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [23]:
# Step 2 — Logistic Regression Parameter Grid

# We don't want an unnecessarily large search space. We'll tune the most impactful hyperparameters.

logistic_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

In [24]:
# Step 3 — Grid Search for Logistic Regression

logistic_grid = GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=logistic_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

logistic_grid.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is disp

In [25]:
# Step 4 — Best Parameters

print("=" * 60)
print("BEST PARAMETERS")
print("=" * 60)

print(logistic_grid.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(round(logistic_grid.best_score_, 4))

BEST PARAMETERS
{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}

Best Cross-Validation F1 Score:
0.5966


In [26]:
best_logistic = logistic_grid.best_estimator_

logistic_tuned_result = evaluate_model(
    best_logistic,
    X_train,
    X_test,
    y_train,
    y_test
)

pd.DataFrame([logistic_tuned_result]).drop(columns="Model Object")

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,LogisticRegression,0.800995,0.640719,0.572193,0.60452,0.835365


## Hyperparameter Tuning Utility

To avoid repetitive code, a reusable helper function is created for performing Grid Search with Stratified K-Fold Cross-Validation.

This function can be used to tune multiple machine learning models using different hyperparameter grids while maintaining a consistent evaluation process.

In [27]:
def tune_model(
    model,
    param_grid,
    X_train,
    y_train,
    cv,
    scoring="f1"
):
    """
    Perform Grid Search hyperparameter tuning.
    """

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    print("=" * 60)
    print("BEST PARAMETERS")
    print("=" * 60)
    print(grid_search.best_params_)

    print("\nBest Cross-Validation Score:")
    print(round(grid_search.best_score_, 4))

    return grid_search.best_estimator_, grid_search

## Tune Gradient Boosting

Gradient Boosting builds trees sequentially, where each tree attempts to correct the errors of the previous one.

The following hyperparameters are tuned:

- Number of boosting stages (`n_estimators`)
- Learning rate
- Maximum tree depth
- Subsample ratio

In [ ]:
# Step 1 — Parameter Grid

gradient_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0]
}

In [29]:
# Step 2 — Tune Model

best_gradient_model, gradient_grid = tune_model(
    GradientBoostingClassifier(random_state=42),
    gradient_param_grid,
    X_train,
    y_train,
    cv_strategy
)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
BEST PARAMETERS
{'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}

Best Cross-Validation Score:
0.5842


In [30]:
# Step 3 — Evaluate Tuned Model

gradient_tuned_result = evaluate_model(
    best_gradient_model,
    X_train,
    X_test,
    y_train,
    y_test
)

pd.DataFrame([gradient_tuned_result]).drop(columns="Model Object")

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,GradientBoostingClassifier,0.782516,0.60303,0.532086,0.565341,0.830418


### Observation

Gradient Boosting was optimized using Grid Search with Stratified 5-Fold Cross-Validation.

The optimal hyperparameters increased the complexity of the model by using deeper trees and a larger number of boosting stages.

Although the tuned model achieved the highest cross-validation F1-score during training, its performance on the independent test set did not improve over the baseline model.

The tuned model showed a slight increase in Recall but experienced reductions in Accuracy, Precision, F1-Score, and ROC-AUC.

Therefore, the baseline Gradient Boosting model remains the preferred configuration for this dataset.

## Tune AdaBoost

AdaBoost (Adaptive Boosting) combines multiple weak learners to create a stronger classifier by giving higher importance to previously misclassified samples.

The following hyperparameters are tuned:

- Number of estimators
- Learning rate

The objective is to identify the configuration that provides the best predictive performance while maintaining good generalization.

In [31]:
# Step 1 — Parameter Grid

adaboost_param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
}

In [32]:
# Step 2 — Tune AdaBoost

best_adaboost_model, adaboost_grid = tune_model(
    AdaBoostClassifier(random_state=42),
    adaboost_param_grid,
    X_train,
    y_train,
    cv_strategy
)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
BEST PARAMETERS
{'learning_rate': 1.0, 'n_estimators': 100}

Best Cross-Validation Score:
0.5888


In [33]:
# Step 3 — Evaluate Tuned AdaBoost

adaboost_tuned_result = evaluate_model(
    best_adaboost_model,
    X_train,
    X_test,
    y_train,
    y_test
)

pd.DataFrame([adaboost_tuned_result]).drop(columns="Model Object")

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,AdaBoostClassifier,0.793888,0.643836,0.502674,0.564565,0.840686


## Part G — Handle Class Imbalance using SMOTE

Customer churn datasets often exhibit class imbalance, where the number of retained customers is significantly higher than the number of churned customers.

Machine learning models trained on such datasets may become biased toward the majority class, resulting in lower Recall for the minority class.

To address this issue, the Synthetic Minority Oversampling Technique (SMOTE) is applied **only to the training dataset**.

SMOTE generates synthetic samples of the minority class without modifying the testing dataset, ensuring a fair evaluation of model performance.

In [34]:
# Step 1 — Import SMOTE

from imblearn.over_sampling import SMOTE

In [35]:
# Step 2 — Apply SMOTE

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [36]:
# Step 3 — Verify the New Distribution

print("=" * 60)
print("CLASS DISTRIBUTION BEFORE SMOTE")
print("=" * 60)

print(y_train.value_counts())

print()

print("=" * 60)
print("CLASS DISTRIBUTION AFTER SMOTE")
print("=" * 60)

print(y_train_smote.value_counts())

CLASS DISTRIBUTION BEFORE SMOTE
Churn
0    4130
1    1495
Name: count, dtype: int64

CLASS DISTRIBUTION AFTER SMOTE
Churn
0    4130
1    4130
Name: count, dtype: int64


In [37]:
# Step 4 — Retrain the Best Models

smote_models = {
    "Logistic Regression": LogisticRegression(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    )
}

In [38]:
# Step 5 — Train on SMOTE Data

smote_results = []

for name, model in smote_models.items():

    print(f"Training {name} with SMOTE...")

    result = evaluate_model(
        model,
        X_train_smote,
        X_test,
        y_train_smote,
        y_test
    )

    smote_results.append(result)

Training Logistic Regression with SMOTE...
Training Gradient Boosting with SMOTE...
Training AdaBoost with SMOTE...


In [39]:
# Step 6 — Create Results Table

smote_results_df = pd.DataFrame(smote_results)

smote_results_df = (
    smote_results_df
    .drop(columns="Model Object")
    .sort_values(
        by=["F1-Score", "ROC-AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

smote_results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,GradientBoostingClassifier,0.746979,0.516484,0.754011,0.613043,0.829081
1,AdaBoostClassifier,0.728500,0.493266,0.783422,0.605372,0.829837
2,LogisticRegression,0.742715,0.511236,0.729947,0.601322,0.827452


## Final Model Selection

Three optimization strategies were evaluated:

1. Baseline models
2. Hyperparameter tuning using Grid Search
3. SMOTE-based class imbalance handling

Hyperparameter tuning did not improve the performance of the shortlisted models on the independent test dataset, indicating that the default model configurations already generalized well.

Applying SMOTE significantly improved the Recall of all models by balancing the minority class during training. Although Accuracy and Precision decreased, the F1-Score improved for the boosting-based models.

Among all evaluated configurations, **Gradient Boosting trained on the SMOTE-balanced dataset** achieved the highest F1-Score (0.6130) while maintaining a competitive ROC-AUC. Since churn prediction prioritizes correctly identifying customers likely to churn, this model provides the best balance between Precision and Recall.

Therefore, **Gradient Boosting with SMOTE** was selected as the final model for this project.